# Cross-Participant EEGBCI Decoding with `coco-pipe`

The trajectory tutorials show how left- and right-hand execution evolve in neural state space. This notebook asks a stricter predictive question:

> **Can left- versus right-hand execution be decoded in a participant who contributed no labeled trials to classifier training?**

We compare two representations under the same leave-one-participant-out protocol:

1. **Sensors:** sliding logistic regression directly on EEG channels.
2. **Aligned PCA:** the same classifier after fold-local temporal Procrustes alignment.

The comparison is designed around leakage prevention. Scaling, PCA, reference-template fitting, alignment, and classification all happen independently inside each outer fold. The held-out participant's labels are used only for final scoring.

<div class="alert alert-secondary">
<b>🗺️ Analysis roadmap:</b><br>
<ol style="margin-bottom: 0; margin-top: 5px;">
  <li>Define the predictive question and inferential unit.</li>
  <li>Load the native sensor-time representation.</li>
  <li>Verify target encoding and participant-level class balance.</li>
  <li>Declare leave-one-participant-out cross-validation.</li>
  <li>Configure fold-local sliding temporal decoding.</li>
  <li>Compare sensors with transductively aligned PCA.</li>
  <li>Run and audit every outer fold.</li>
  <li>Inspect time-resolved generalization.</li>
  <li>Examine held-out-participant variability.</li>
  <li>Export the full analysis and structured HTML report.</li>
</ol>
</div>

## 0. Setup & Configuration

All scientific choices are declared before data loading. The notebook performs the complete analysis directly; it does not call the headless analysis function. The final report renderer is shared with the companion script so both entry points produce exactly the same ten-section report.

### 0.1. Environment & imports

`coco-pipe` owns the cross-validation engine, fold-local preprocessing, temporal estimator, alignment, result containers, and plotting. The repository helper only loads the prepared EEGBCI epochs and records provenance.

In [ ]:
# --- Standard library -------------------------------------------------------
import os
import warnings
from pathlib import Path

# --- Numerical and plotting libraries ---------------------------------------
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

# --- coco-pipe decoding ------------------------------------------------------
from coco_pipe.decoding import (
    CVConfig,
    Experiment,
    ExperimentConfig,
    TemporalAlignmentConfig,
    TemporalDecoderConfig,
)
from coco_pipe.decoding.configs import ClassicalModelConfig
from coco_pipe.viz.interactive.decoding import plot_temporal_score_curve
from coco_pipe.viz.theme import set_coco_theme

# --- Repository helpers ------------------------------------------------------
from pca_neural_trajectories import (
    LABEL_NAMES,
    load_eegbci_container,
    setup_data_bids,
    write_manifest,
)
from scripts.analysis_eegbci_decoding import build_decoding_report

### 0.2. Analysis parameters & output contract

The default tutorial uses the first ten compatible EEGBCI participants. Subjects 88, 92, and 100 are excluded because their stored epochs have incompatible sampling rates. The full-cohort execution previously used every other compatible participant.

<div class="alert alert-info">
<b>⚙️ Environment overrides:</b><br>
Use <code>EEG_N_SUBJECTS</code>, <code>EEG_DECODING_N_JOBS</code>, <code>EEG_BIDS_ROOT</code>, and <code>EEG_DECODING_OUTPUT</code> to change runtime settings without editing analysis cells. Set <code>EEG_PREPARE_DATA=1</code> only when preprocessing should be requested explicitly.
</div>

In [ ]:
set_coco_theme(mode="paper", colorblind=True)

SEED = 42
CONDITIONS = (3, 4)
ANALYSIS_WINDOW = (-0.2, 1.0)
CHANCE_LEVEL = 0.5
N_COMPONENTS = int(os.getenv("EEG_DECODING_COMPONENTS", "30"))
N_JOBS = int(os.getenv("EEG_DECODING_N_JOBS", "-1"))
N_SUBJECTS = int(os.getenv("EEG_N_SUBJECTS", "10"))
BIDS_ROOT = Path(os.getenv("EEG_BIDS_ROOT", "PhysioNet_EEGBCI/BIDS"))
OUTPUT = Path(os.getenv("EEG_DECODING_OUTPUT", "outputs/tutorial_eegbci_decoding"))
FIGURES_DIR = OUTPUT / "figures"
RESULTS_DIR = OUTPUT / "experiment_results"
PREPARE_DATA = os.getenv("EEG_PREPARE_DATA", "0") == "1"

EXCLUDED_SUBJECTS = {88, 92, 100}
available = [subject for subject in range(1, 110) if subject not in EXCLUDED_SUBJECTS]
SUBJECTS_REQUESTED = available[:N_SUBJECTS]
SUBJECTS = tuple(f"{subject:03d}" for subject in SUBJECTS_REQUESTED)
REPRESENTATION_COLORS = {
    "Sensors": "#1b9e77",
    "Aligned PCA": "#2a78d6",
}

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Participants requested: {SUBJECTS}")
print(f"BIDS input: {BIDS_ROOT}")
print(f"Output bundle: {OUTPUT}")

## Step 1. Define the Predictive Question

The question is **cross-participant generalization**, not whether a model can classify held-out trials from a familiar participant. This distinction determines the entire validation design.

- **Observation:** one epoched movement trial.
- **Target:** left-hand (`0`) versus right-hand (`1`) execution.
- **Inferential unit:** participant.
- **Generalization target:** a participant whose labeled trials were absent during training.

<div class="alert alert-danger">
<b>🚫 Why random trial splitting is invalid:</b><br>
Trials from one participant share anatomy, sensor placement, preprocessing history, and stable individual signal structure. Putting their trials in both training and test sets can let the classifier recognize the participant rather than learn a pattern that transfers across participants.
</div>

## Step 2. Load the Native Sensor-Time Representation

`coco-pipe` accepts the original `(trial, channel, time)` array directly, so no custom time-binning helper is needed. We load the two execution conditions over −0.2–1.0 s and apply the pre-cue −0.2–0.0 s sensor baseline.

<div class="alert alert-warning">
<b>🧠 Preprocessing boundary:</b><br>
The introductory tutorial prepares the BIDS derivatives. This notebook does not silently download or preprocess data. Set <code>PREPARE_DATA=True</code> only when that external work is explicitly intended.
</div>

In [ ]:
if PREPARE_DATA:
    setup_data_bids(
        subjects=SUBJECTS_REQUESTED,
        runs=list(range(3, 15)),
        root=BIDS_ROOT,
    )
if not BIDS_ROOT.exists():
    raise FileNotFoundError(
        f"EEGBCI BIDS data were not found at {BIDS_ROOT}. "
        "Run the introductory tutorial first."
    )

container = load_eegbci_container(
    BIDS_ROOT,
    subjects=SUBJECTS,
    runs=tuple(range(3, 15)),
    conditions=CONDITIONS,
    tmin=ANALYSIS_WINDOW[0],
    tmax=ANALYSIS_WINDOW[1],
    baseline=(-0.2, 0.0),
)
X = np.asarray(container.X, dtype=np.float32)
times = np.asarray(container.coords["time"], dtype=float)
condition = np.asarray(container.y, dtype=int)
subject_ids = np.asarray(container.coords["subject"]).astype(str)
trial_ids = np.asarray(container.ids).astype(str)
y = np.where(condition == CONDITIONS[0], 0, 1)

ANALYZED_SUBJECTS = sorted(np.unique(subject_ids).tolist())
if len(ANALYZED_SUBJECTS) < 2:
    raise RuntimeError("Cross-participant decoding requires at least two participants.")
if N_COMPONENTS > X.shape[1]:
    raise ValueError(f"N_COMPONENTS={N_COMPONENTS} exceeds {X.shape[1]} channels.")

In [ ]:
data_summary = pd.Series({
    "trials": X.shape[0],
    "channels": X.shape[1],
    "time samples": X.shape[2],
    "participants": len(ANALYZED_SUBJECTS),
    "first time (s)": float(times[0]),
    "last time (s)": float(times[-1]),
})
data_summary.to_frame("value")

## Step 3. Verify the Target and Participant-Level Class Balance

The target encoding is fixed before model fitting: label 3 becomes left hand (`0`) and label 4 becomes right hand (`1`). We document the number of trials in every participant × class cell.

Balanced accuracy is the mean recall across the two classes, so each class contributes equally to the score even if a participant has modestly unequal trial counts. The classifier also uses `class_weight='balanced'`; those weights are estimated from the **training fold only**.

<div class="alert alert-success">
<b>✅ No test-set resampling:</b><br>
Class balance is handled by the metric and training-fold weights. Test trials are never duplicated, removed, or used to choose a training weight.
</div>

In [ ]:
trial_counts = (
    pd.DataFrame({
        "subject": subject_ids,
        "condition": condition,
        "condition_name": [LABEL_NAMES[value] for value in condition],
    })
    .groupby(["subject", "condition", "condition_name"], as_index=False)
    .size()
    .rename(columns={"size": "n_trials"})
)
display(trial_counts)

trial_counts.pivot(index="subject", columns="condition_name", values="n_trials")

## Step 4. Declare Leave-One-Participant-Out Cross-Validation

`CVConfig(strategy='leave_one_group_out')` creates one outer fold per participant. Every trial from the held-out participant becomes test data; every trial from all other participants becomes training data.

The same participant groups are passed to both representations, so sensor and aligned scores are naturally paired by held-out participant. We will verify the generated splits after fitting rather than assuming the configuration was applied correctly.

## Step 5. Configure Fold-Local Sliding Temporal Decoding

At each latency, `wrapper='sliding'` fits a separate logistic-regression classifier using the channel or component values at that time point. This yields a time-resolved generalization curve.

`use_scaler=True` instructs `coco-pipe` to fit standardization inside each training fold. The held-out participant is transformed with those training-fold parameters. Fitting a `StandardScaler` once on all participants before LOSO would leak test-distribution information.

<div class="alert alert-warning">
<b>📈 Multiple-latency caution:</b><br>
Every plotted time point is a separate fitted model. The highest point is a useful descriptive summary, but it is not automatically a family-wise-error-corrected significance result.
</div>

In [ ]:
decoder = TemporalDecoderConfig(
    wrapper="sliding",
    base=ClassicalModelConfig(
        estimator="LogisticRegression",
        params={"class_weight": "balanced", "max_iter": 2000},
    ),
    n_jobs=1,
    verbose=False,
)
sensor_config = ExperimentConfig(
    task="classification",
    models={"Logistic regression": decoder},
    metrics=["balanced_accuracy"],
    cv=CVConfig(strategy="leave_one_group_out", shuffle=False),
    use_scaler=True,
    random_state=SEED,
    n_jobs=N_JOBS,
    verbose=False,
)
sensor_config

## Step 6. Compare Sensors with Fold-Local Aligned PCA

The sensor experiment uses the native channels. For aligned PCA, we copy the sensor configuration and change only `temporal_alignment`. This makes the representation the sole experimental difference.

Within each outer fold, alignment:

1. fits the shared PCA reference and temporal template using training participants;
2. fits a participant PCA to the unseen participant's **unlabeled** trials;
3. estimates an orthogonal rotation/reflection into the training reference; and
4. passes the aligned time-resolved features into the same fold-local scaler and classifier.

<div class="alert alert-danger">
<b>🔒 Transductive scope:</b><br>
Aligned performance assumes an unlabeled calibration batch from the new participant. No held-out labels are used for adaptation, but this is not zero-calibration inductive decoding.
</div>

In [ ]:
aligned_config = sensor_config.model_copy(deep=True)
aligned_config.temporal_alignment = TemporalAlignmentConfig(
    enabled=True,
    n_components=N_COMPONENTS,
    adaptation="transductive",
)
experiments = {
    "Sensors": sensor_config,
    "Aligned PCA": aligned_config,
}

pd.DataFrame([
    {
        "Representation": name,
        "alignment_enabled": config.temporal_alignment.enabled,
        "alignment_components": (
            config.temporal_alignment.n_components
            if config.temporal_alignment.enabled else None
        ),
        "adaptation": (
            config.temporal_alignment.adaptation
            if config.temporal_alignment.enabled else None
        ),
        "scaler_inside_fold": config.use_scaler,
        "cv": config.cv.strategy,
    }
    for name, config in experiments.items()
])

## Step 7. Run and Preserve Every Outer Fold

Both experiments receive the same epochs, labels, participant groups, trial identifiers, and scientific time axis. `ExperimentResult` retains the fold scores, predictions, splits, fit diagnostics, and configuration needed for later audit.

We export each complete result object rather than saving only the final mean curve. This allows future checks without refitting hundreds of temporal classifiers.

In [ ]:
results = {}
temporal_frames = []
fold_frames = []
diagnostic_frames = []

for representation, config in experiments.items():
    print(f"Running {representation} ...")
    result = Experiment(config).run(
        X,
        y,
        groups=subject_ids,
        sample_ids=trial_ids,
        observation_level="epoch",
        inferential_unit="subject",
        time_axis=times,
    )
    results[representation] = result

    temporal = result.get_temporal_score_summary()
    temporal["Estimator"] = temporal["Model"]
    temporal["Representation"] = representation
    temporal["Model"] = representation
    temporal_frames.append(temporal)

    folds = result.get_detailed_scores()
    folds["Estimator"] = folds["Model"]
    folds["Representation"] = representation
    fold_frames.append(folds)

    diagnostics = result.get_fit_diagnostics()
    diagnostics["Estimator"] = diagnostics["Model"]
    diagnostics["Representation"] = representation
    diagnostic_frames.append(diagnostics)

    result.export(
        RESULTS_DIR / representation.lower().replace(" ", "_"),
        config=config.model_dump(),
        formats=("csv",),
    )

temporal_scores = pd.concat(temporal_frames, ignore_index=True)
fold_scores = pd.concat(fold_frames, ignore_index=True)
fit_diagnostics = pd.concat(diagnostic_frames, ignore_index=True)

### Audit the generated splits

Configuration is not proof. We derive a fold audit from the stored sensor-result splits and verify that every fold contains exactly one test participant with no participant overlap between training and test sets. The aligned experiment uses the identical outer splitter.

In [ ]:
split_rows = results["Sensors"].get_splits()
audit_records = []
for fold in sorted(split_rows["Fold"].unique()):
    fold_rows = split_rows[split_rows["Fold"] == fold]
    train_rows = fold_rows[fold_rows["Set"] == "train"]
    test_rows = fold_rows[fold_rows["Set"] == "test"]
    train_subjects = sorted(train_rows["Group"].astype(str).unique())
    test_subjects = sorted(test_rows["Group"].astype(str).unique())
    overlap = sorted(set(train_subjects) & set(test_subjects))
    audit_records.append({
        "Fold": int(fold),
        "held_out_subject": ", ".join(test_subjects),
        "n_train_subjects": len(train_subjects),
        "n_test_subjects": len(test_subjects),
        "n_train_trials": len(train_rows),
        "n_test_trials": len(test_rows),
        "subject_overlap": ", ".join(overlap),
        "leakage_free": len(overlap) == 0 and len(test_subjects) == 1,
    })

split_audit = pd.DataFrame(audit_records)
if not split_audit["leakage_free"].all():
    raise RuntimeError("The outer-fold audit found participant overlap.")
split_audit

## Step 8. Inspect Time-Resolved Cross-Participant Generalization

The line is mean balanced accuracy across held-out participants; the ribbon is the fold-level standard deviation returned by `coco-pipe`. The dotted horizontal line marks binary chance and the vertical line marks movement-cue onset.

Peak latency and accuracy summarize the displayed curve. They remain descriptive: selecting the maximum across many correlated latencies introduces optimism unless evaluated with a dedicated corrected inferential procedure. Sustained post-cue performance is therefore examined in Step 9.

In [ ]:
temporal_figure = plot_temporal_score_curve(
    temporal_scores,
    metric="balanced_accuracy",
    title="Left- versus right-hand execution: LOSO decoding",
    colors=REPRESENTATION_COLORS,
)
temporal_figure.add_hline(y=CHANCE_LEVEL, line_dash="dot", line_color="#777777")
temporal_figure.add_vline(x=0, line_color="#999999")
temporal_figure.update_yaxes(title_text="balanced accuracy")
temporal_figure.update_xaxes(title_text="time from movement cue (s)")
temporal_figure.show()

In [ ]:
peak_summary = temporal_scores.loc[
    temporal_scores.groupby("Representation")["Mean"].idxmax(),
    ["Representation", "Time", "Mean", "Std"],
].sort_values("Mean", ascending=False)
peak_summary = peak_summary.rename(columns={
    "Time": "peak_time_s",
    "Mean": "peak_balanced_accuracy",
    "Std": "peak_fold_std",
})
peak_summary.round(3)

## Step 9. Examine Held-Out-Participant Variability

A group mean can hide participants with qualitatively different decoding profiles. We therefore retain one temporal curve per LOSO fold and label it with the held-out participant.

Two predeclared post-cue summaries reduce each fold to interpretable scalars:

1. **Post-cue mean balanced accuracy:** average accuracy from 0–1 s.
2. **AUC above chance:** integral of `(balanced accuracy − 0.5)` over 0–1 s.

Aligned-minus-sensor differences are paired within the same held-out participant. These values diagnose consistency and effect direction; no p-value is attached.

In [ ]:
metric_rows = fold_scores[
    (fold_scores["Metric"] == "balanced_accuracy")
    & fold_scores["Time"].notna()
]
held_out_by_fold = split_audit.set_index("Fold")["held_out_subject"].to_dict()
fold_summary_records = []
for (representation, fold), rows in metric_rows.groupby(["Representation", "Fold"]):
    rows = rows.sort_values("Time")
    active_rows = rows[rows["Time"] >= 0]
    peak_index = rows["Value"].idxmax()
    fold_summary_records.append({
        "Representation": representation,
        "Fold": int(fold),
        "held_out_subject": held_out_by_fold[int(fold)],
        "peak_time_s": float(rows.loc[peak_index, "Time"]),
        "peak_balanced_accuracy": float(rows.loc[peak_index, "Value"]),
        "postcue_mean_balanced_accuracy": float(active_rows["Value"].mean()),
        "postcue_auc_above_chance": float(np.trapezoid(
            active_rows["Value"] - CHANCE_LEVEL, active_rows["Time"]
        )),
    })

fold_summary = pd.DataFrame(fold_summary_records)
representation_summary = (
    fold_summary.groupby("Representation")
    .agg(
        n_folds=("Fold", "nunique"),
        peak_ba_mean=("peak_balanced_accuracy", "mean"),
        peak_ba_std=("peak_balanced_accuracy", "std"),
        postcue_ba_mean=("postcue_mean_balanced_accuracy", "mean"),
        postcue_ba_std=("postcue_mean_balanced_accuracy", "std"),
        postcue_auc_mean=("postcue_auc_above_chance", "mean"),
        postcue_auc_std=("postcue_auc_above_chance", "std"),
    )
    .reset_index()
)
display(fold_summary)
display(representation_summary)

In [ ]:
paired = fold_summary.pivot(
    index=["Fold", "held_out_subject"],
    columns="Representation",
    values=[
        "peak_balanced_accuracy",
        "postcue_mean_balanced_accuracy",
        "postcue_auc_above_chance",
    ],
)
paired_fold_differences = paired.index.to_frame(index=False)
for metric in (
    "peak_balanced_accuracy",
    "postcue_mean_balanced_accuracy",
    "postcue_auc_above_chance",
):
    paired_fold_differences[f"{metric}_aligned_minus_sensors"] = (
        paired[(metric, "Aligned PCA")].to_numpy()
        - paired[(metric, "Sensors")].to_numpy()
    )
paired_fold_differences

In [ ]:
fold_heatmaps = make_subplots(
    rows=1, cols=2, subplot_titles=("Sensors", "Aligned PCA"), horizontal_spacing=0.12
)
for column, representation in enumerate(("Sensors", "Aligned PCA"), start=1):
    rows = metric_rows[metric_rows["Representation"] == representation]
    matrix = rows.pivot(index="Fold", columns="Time", values="Value").sort_index()
    fold_labels = [f"sub-{held_out_by_fold[int(fold)]}" for fold in matrix.index]
    fold_heatmaps.add_trace(
        go.Heatmap(
            z=matrix.to_numpy(),
            x=matrix.columns.to_numpy(dtype=float),
            y=fold_labels,
            zmin=0, zmax=1, colorscale="Viridis",
            colorbar={"title": "BA"} if column == 2 else None,
            showscale=column == 2,
        ),
        row=1, col=column,
    )
fold_heatmaps.update_xaxes(title_text="time from movement cue (s)")
fold_heatmaps.update_yaxes(title_text="held-out participant", row=1, col=1)
fold_heatmaps.update_layout(
    title="Balanced accuracy for every held-out participant",
    height=max(520, 32 * len(ANALYZED_SUBJECTS) + 250),
    width=1050,
)
fold_heatmaps.show()

In [ ]:
fold_summary_figure = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Post-cue mean balanced accuracy", "Post-cue AUC above chance"),
)
for column, metric in enumerate(
    ("postcue_mean_balanced_accuracy", "postcue_auc_above_chance"), start=1
):
    for representation in ("Sensors", "Aligned PCA"):
        rows = fold_summary[fold_summary["Representation"] == representation]
        fold_summary_figure.add_trace(
            go.Box(
                x=[representation] * len(rows), y=rows[metric],
                name=representation, marker_color=REPRESENTATION_COLORS[representation],
                boxpoints="all", jitter=0.25, pointpos=0,
                showlegend=column == 1, legendgroup=representation,
            ),
            row=1, col=column,
        )
fold_summary_figure.add_hline(
    y=CHANCE_LEVEL, line_dash="dot", line_color="#777777", row=1, col=1
)
fold_summary_figure.add_hline(
    y=0, line_dash="dot", line_color="#777777", row=1, col=2
)
fold_summary_figure.update_layout(
    title="Held-out-participant post-cue summaries", height=500, width=1000
)
fold_summary_figure.show()

## Step 10. Export the Complete Analysis and Structured Report

A reproducible decoding analysis must preserve more than the group-mean curve. The output bundle includes:

- participant × class trial counts;
- the outer-fold leakage audit;
- mean and fold-level temporal scores;
- peak, post-cue, and paired fold summaries;
- fit diagnostics;
- complete serialized `ExperimentResult` objects and their tidy exports;
- interactive and static figures;
- a provenance manifest; and
- a self-contained ten-step HTML report with the same explanations as this notebook.

The notebook and script share only the report renderer. Every analysis result supplied to that renderer was computed explicitly in the cells above.

In [ ]:
tables = {
    "trial_counts": trial_counts,
    "split_audit": split_audit,
    "temporal_scores": temporal_scores,
    "fold_scores": fold_scores,
    "peak_summary": peak_summary,
    "fold_summary": fold_summary,
    "representation_summary": representation_summary,
    "paired_fold_differences": paired_fold_differences,
    "fit_diagnostics": fit_diagnostics,
}
figures = {
    "temporal_decoding": temporal_figure,
    "fold_heatmaps": fold_heatmaps,
    "fold_summaries": fold_summary_figure,
}
for name, table in tables.items():
    table.to_csv(OUTPUT / f"{name}.csv", index=False)

In [ ]:
static_export_error = None
for name, figure in figures.items():
    figure.write_html(FIGURES_DIR / f"{name}.html", include_plotlyjs="cdn")
    if static_export_error is None:
        try:
            figure.write_image(FIGURES_DIR / f"{name}.png", scale=2)
            figure.write_image(FIGURES_DIR / f"{name}.svg")
        except Exception as error:
            static_export_error = f"{type(error).__name__}: {error}"
            warnings.warn(
                "Static Plotly export is unavailable; continuing with HTML figures. "
                "Check the Kaleido browser installation.",
                stacklevel=2,
            )

np.savez_compressed(
    OUTPUT / "analysis_metadata.npz",
    times=times, condition=condition, target=y, subjects=subject_ids, trial_ids=trial_ids,
)

In [ ]:
report_context = {
    "n_subjects": len(ANALYZED_SUBJECTS),
    "n_trials": len(X),
    "n_channels": X.shape[1],
    "n_times": X.shape[2],
    "time_start": float(times[0]),
    "time_stop": float(times[-1]),
    "chance_level": CHANCE_LEVEL,
    "n_components": N_COMPONENTS,
    "static_exports_complete": static_export_error is None,
}
report_asset_mode = build_decoding_report(
    OUTPUT, context=report_context, tables=tables, figures=figures
)

manifest = {
    "subjects_requested": SUBJECTS_REQUESTED,
    "subjects_analyzed": ANALYZED_SUBJECTS,
    "bids_root": str(BIDS_ROOT),
    "conditions": list(CONDITIONS),
    "analysis_window": list(ANALYSIS_WINDOW),
    "target_mapping": {"0": LABEL_NAMES[3], "1": LABEL_NAMES[4]},
    "shape": list(X.shape),
    "cv": "leave_one_group_out",
    "metric": "balanced_accuracy",
    "chance_level": CHANCE_LEVEL,
    "n_components": N_COMPONENTS,
    "alignment_adaptation": "transductive",
    "random_state": SEED,
    "n_jobs": N_JOBS,
    "static_figure_exports_complete": static_export_error is None,
    "static_figure_export_error": static_export_error,
    "report_asset_mode": report_asset_mode,
}
write_manifest(OUTPUT / "analysis_manifest.json", manifest, status="complete")
print(f"Saved EEGBCI decoding analysis → {OUTPUT}")

In [ ]:
saved_files = sorted(
    str(path.relative_to(OUTPUT))
    for path in OUTPUT.rglob("*")
    if path.is_file()
)
pd.DataFrame({"saved_file": saved_files})

## Conclusions & Interpretation Checklist

Before making a decoding claim, verify the full chain of evidence:

- **Participant separation:** every outer fold must have zero train/test participant overlap.
- **Fold-local preprocessing:** scaling and alignment must be fitted after the outer split.
- **Inferential unit:** folds represent held-out participants, not independent time points or trials.
- **Temporal multiplicity:** a descriptive peak is not a corrected significance test.
- **Participant variability:** the mean curve should be read alongside fold heatmaps and paired summaries.
- **Calibration scope:** aligned PCA uses unlabeled held-out-participant trials and is therefore transductive.

<div class="alert alert-success">
<b>🎯 Main takeaway:</b><br>
Compare sensors and aligned PCA through paired held-out-participant behavior and sustained temporal performance. A single group-level peak is insufficient to establish that alignment improves generalization.
</div>

## Running the Same Analysis Headlessly

The companion script repeats the same computations, exports the complete `ExperimentResult` objects, and invokes the same report renderer:

```bash
python scripts/analysis_eegbci_decoding.py
```

For a small validation run, use `--subjects 1 2 --n-jobs 1`. Add `--prepare` only when the script should explicitly prepare the requested BIDS data first.